# Modelling and Evaluation**DCS 404 — Machine Learning and Artificial Intelligence**Sanskriti Acharya and Sweta SharmaThe research question, stated plainly:> Can an address's network of connections reveal fraud that its own features cannot?To answer it we train three tiers on an identical split, changing nothing but thefeature set:1. **Majority class** — always predicts "licit". The trivial baseline.2. **Behaviour only** — what the address does on its own.3. **Behaviour + graph** — plus who it is connected to.The difference between tier 2 and tier 3 *is* the answer. Everything else is heldconstant so that difference means something.The training code itself lives in `src/train.py`; this notebook imports it ratherthan reimplementing it, so what is measured here is exactly what ships in the app.

In [ ]:
import sysfrom pathlib import PathROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ROOT))import matplotlib.pyplot as pltimport numpy as npimport pandas as pdimport seaborn as snsfrom sklearn.metrics import (    ConfusionMatrixDisplay, classification_report, confusion_matrix,    precision_recall_curve, roc_curve, average_precision_score, roc_auc_score,)from src.features import BEHAVIOUR_FEATURES, GRAPH_FEATURESfrom src.train import temporal_split, run_tier, RANDOM_STATEsns.set_theme(style="whitegrid", palette="muted")plt.rcParams["figure.dpi"] = 110pd.set_option("display.width", 140)FIGURES = ROOT / "reports" / "figures"FIGURES.mkdir(parents=True, exist_ok=True)

## 1. The splitAddresses are ordered by when they first appeared on chain; the earliest 70% trainand the most recent 30% test.A random split would be easier and would score better, which is exactly theproblem. Scam operations run clusters of wallets that behave alike, and a randomsplit cheerfully puts one wallet from a cluster in training and its sibling intest — the model then recognises the cluster rather than generalising. Splittingby time mimics the real task: learn from what is known, then score addresses thatcame later.

In [ ]:
full = pd.read_csv(ROOT / "data" / "processed" / "features_full.csv")train, test = temporal_split(full)print(f"train: {len(train):>5} addresses, {int(train['label'].sum()):>4} illicit "      f"({train['label'].mean():.1%})")print(f"test:  {len(test):>5} addresses, {int(test['label'].sum()):>4} illicit "      f"({test['label'].mean():.1%})")active = full[full["first_block"] > 0]print(f"\ntrain covers blocks {train[train.first_block > 0].first_block.min():,.0f} "      f"to {train[train.first_block > 0].first_block.max():,.0f}")print(f"test  covers blocks {test[test.first_block > 0].first_block.min():,.0f} "      f"to {test[test.first_block > 0].first_block.max():,.0f}")

## 2. Why F1 on the illicit classThe metric has to be chosen before the results are seen, and justified.**Accuracy is unusable.** With roughly a quarter of addresses illicit, predicting"licit" every time scores about 77%. Any metric that rewards that is measuring thewrong thing.**The two errors cost different amounts.** A missed scam address means fraudcontinues and victims keep paying. A false alarm costs an analyst a few minutesof review. Recall matters more than precision here — but not infinitely, becausea model that flags everything is just as useless as one that flags nothing.**F1 on the illicit class** is the harmonic mean of precision and recall, so itonly rises when both are decent. We report **PR-AUC** alongside it, whichsummarises that trade-off across every threshold rather than only at 0.5, and isthe right companion for an imbalanced problem.

## 3. Tier 1 — the trivial baselineThe spec is explicit that this must be built and reported, not assumed. It is hereso every later number has something honest to be compared against.

In [ ]:
from sklearn.dummy import DummyClassifierfrom src.train import evaluatedummy = DummyClassifier(strategy="most_frequent")dummy.fit(train[BEHAVIOUR_FEATURES], train["label"])baseline = evaluate("majority class", test["label"].to_numpy(),                    dummy.predict(test[BEHAVIOUR_FEATURES]))print(f"accuracy  {baseline['accuracy']:.3f}   <- looks respectable")print(f"precision {baseline['precision']:.3f}")print(f"recall    {baseline['recall']:.3f}")print(f"F1        {baseline['f1']:.3f}   <- and here is the truth")

An F1 of 0 against an accuracy near 0.77, from a model that has learned nothing atall. This single line is the clearest argument for the metric choice above.

## 4. Tiers 2 and 3Both tiers fit a logistic regression and a random forest, each as a fullscikit-learn pipeline with the scaler inside it. Hyperparameters are tuned by5-fold cross-validation **on the training split only** — the test split is touchedonce, at scoring time, and never used to make a choice.

In [ ]:
tier2, fitted2 = run_tier("behaviour", BEHAVIOUR_FEATURES, train, test)for row in tier2:    print(f"{row['model']:<32} F1 {row['f1']:.3f}  PR-AUC {row['pr_auc']:.3f}  "          f"(CV F1 {row['cv_f1']:.3f})  {row['best_params']}")

In [ ]:
combined = BEHAVIOUR_FEATURES + GRAPH_FEATUREStier3, fitted3 = run_tier("behaviour+graph", combined, train, test)for row in tier3:    print(f"{row['model']:<32} F1 {row['f1']:.3f}  PR-AUC {row['pr_auc']:.3f}  "          f"(CV F1 {row['cv_f1']:.3f})  {row['best_params']}")

## 5. Results

In [ ]:
baseline_row = {**baseline, "tier": "baseline", "model": "baseline / majority_class",                "cv_f1": np.nan}results = pd.DataFrame([baseline_row] + tier2 + tier3)table = results[["model", "accuracy", "precision", "recall", "f1", "pr_auc", "roc_auc"]].round(3)display(table)results.to_csv(ROOT / "reports" / "metrics.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))order = results["model"].tolist()colours = ["#8c8c8c" if "majority" in m else "#4c72b0" if "behaviour /" in m else "#c44e52"           for m in order]axes[0].barh(order, results["f1"], color=colours)axes[0].set_xlabel("F1 (illicit class)")axes[0].set_title("F1 by tier")for i, v in enumerate(results["f1"]):    axes[0].text(v, i, f" {v:.3f}", va="center", fontsize=9)metrics_long = results.melt(    id_vars="model", value_vars=["precision", "recall", "f1"],    var_name="metric", value_name="score",)sns.barplot(data=metrics_long, y="model", x="score", hue="metric", ax=axes[1])axes[1].set_title("Precision, recall and F1")axes[1].set_ylabel("")plt.tight_layout()plt.savefig(FIGURES / "model_comparison.png", bbox_inches="tight")plt.show()

In [ ]:
best_behaviour = max(r["f1"] for r in tier2)best_combined = max(r["f1"] for r in tier3)lift = best_combined - best_behaviourprint(f"best behaviour-only F1   {best_behaviour:.3f}")print(f"best behaviour+graph F1  {best_combined:.3f}")print(f"lift from graph features {lift:+.3f}")print()print(f"both beat the trivial baseline's F1 of {baseline['f1']:.3f}")

### Answering the research questionThe lift printed above is the project's result. Whichever way it comes out, it isreported as measured — a small or absent lift is a finding, not a failure, andsaying so honestly is worth more than a flattering number.One thing to keep in mind when reading it: our graph is built only from thetransactions of labelled addresses and their immediate counterparties, capped at64 transactions each by the API's rate limit. A denser graph would give thenetwork features more to work with, so this measures the lift available from a*sparse* graph, not the ceiling.

## 6. Where the shipped model gets things wrongAggregate metrics hide the interesting part. This is the error analysis the specasks for.

In [ ]:
best_name = max(tier3, key=lambda r: r["f1"])["model"].split(" / ")[1]best_model = fitted3[best_name]y_true = test["label"].to_numpy()y_pred = best_model.predict(test[combined])y_score = best_model.predict_proba(test[combined])[:, 1]print(f"shipped model: {best_name}\n")print(classification_report(y_true, y_pred, target_names=["licit", "illicit"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))matrix = confusion_matrix(y_true, y_pred)ConfusionMatrixDisplay(matrix, display_labels=["licit", "illicit"]).plot(    ax=axes[0], cmap="Blues", colorbar=False)axes[0].set_title("Confusion matrix")precision, recall, _ = precision_recall_curve(y_true, y_score)axes[1].plot(recall, precision, color="#c44e52")axes[1].axhline(y_true.mean(), ls="--", c="grey",                label=f"random ({y_true.mean():.2f})")axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")axes[1].set_title(f"Precision-Recall (AP = {average_precision_score(y_true, y_score):.3f})")axes[1].legend()fpr, tpr, _ = roc_curve(y_true, y_score)axes[2].plot(fpr, tpr, color="#4c72b0")axes[2].plot([0, 1], [0, 1], ls="--", c="grey")axes[2].set_xlabel("false positive rate"); axes[2].set_ylabel("true positive rate")axes[2].set_title(f"ROC (AUC = {roc_auc_score(y_true, y_score):.3f})")plt.tight_layout()plt.savefig(FIGURES / "evaluation_curves.png", bbox_inches="tight")plt.show()tn, fp, fn, tp = matrix.ravel()print(f"true negatives  {tn:>4}   false positives {fp:>4}")print(f"false negatives {fn:>4}   true positives  {tp:>4}")

In [ ]:
# What do the mistakes look like? Compare the missed scams against the ones caught.errors = test.copy()errors["predicted"] = y_prederrors["probability"] = y_scoremissed = errors[(errors.label == 1) & (errors.predicted == 0)]caught = errors[(errors.label == 1) & (errors.predicted == 1)]false_alarms = errors[(errors.label == 0) & (errors.predicted == 1)]print(f"missed scams: {len(missed)}   caught: {len(caught)}   false alarms: {len(false_alarms)}")print()compare = pd.DataFrame({    "missed (FN)": missed[["n_tx", "n_counterparties", "lifetime_days",                           "total_eth_received", "degree"]].mean(),    "caught (TP)": caught[["n_tx", "n_counterparties", "lifetime_days",                           "total_eth_received", "degree"]].mean(),    "false alarm (FP)": false_alarms[["n_tx", "n_counterparties", "lifetime_days",                                      "total_eth_received", "degree"]].mean(),}).round(2)display(compare)

In [ ]:
print("Transactions available for missed scams vs caught ones:")print(f"  missed: median {missed['n_tx'].median():.0f} transactions, "      f"{(missed['n_tx'] == 0).sum()} with none at all")print(f"  caught: median {caught['n_tx'].median():.0f} transactions")print()plt.figure(figsize=(9, 4))plt.hist(errors[errors.label == 0]["probability"], bins=30, alpha=0.6,         label="licit", color="#4c72b0", density=True)plt.hist(errors[errors.label == 1]["probability"], bins=30, alpha=0.6,         label="illicit", color="#c44e52", density=True)plt.axvline(0.5, ls="--", c="k", lw=1, label="threshold")plt.xlabel("predicted probability of illicit")plt.ylabel("density")plt.title("Where the model is confident, and where it is not")plt.legend()plt.tight_layout()plt.savefig(FIGURES / "probability_distribution.png", bbox_inches="tight")plt.show()

The overlapping middle of that histogram is the honest picture of this model: aband of addresses it genuinely cannot tell apart. In deployment that band is not afailure, it is the queue — which is why the application presents a probability anda review priority rather than a bare verdict.

## 7. What the model relies on

In [ ]:
model = best_model.named_steps["model"]if hasattr(model, "feature_importances_"):    importance = pd.DataFrame({"feature": combined,                               "importance": model.feature_importances_})else:    importance = pd.DataFrame({"feature": combined,                               "importance": np.abs(model.coef_[0])})importance["is_graph"] = importance["feature"].isin(GRAPH_FEATURES)importance = importance.sort_values("importance", ascending=False)top = importance.head(18).iloc[::-1]plt.figure(figsize=(8, 6))plt.barh(top["feature"], top["importance"],         color=["#c44e52" if g else "#4c72b0" for g in top["is_graph"]])plt.xlabel("importance")plt.title("Most important features (red = graph-derived)")plt.tight_layout()plt.savefig(FIGURES / "feature_importance.png", bbox_inches="tight")plt.show()share = importance[importance.is_graph]["importance"].sum() / importance["importance"].sum()print(f"graph features account for {share:.1%} of total importance")print(f"({len(GRAPH_FEATURES)} of {len(combined)} features are graph-derived)")

## 8. Threshold choice0.5 is a default, not a decision. An exchange's triage team can afford to reviewmore addresses in exchange for missing fewer scams, so it is worth showing whatthat trade costs.

In [ ]:
rows = []for threshold in np.arange(0.1, 0.91, 0.05):    predicted = (y_score >= threshold).astype(int)    tp = int(((predicted == 1) & (y_true == 1)).sum())    fp = int(((predicted == 1) & (y_true == 0)).sum())    fn = int(((predicted == 0) & (y_true == 1)).sum())    precision = tp / (tp + fp) if tp + fp else 0.0    recall = tp / (tp + fn) if tp + fn else 0.0    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0    rows.append({"threshold": round(threshold, 2), "flagged": tp + fp,                 "precision": precision, "recall": recall, "f1": f1})sweep = pd.DataFrame(rows)best = sweep.loc[sweep["f1"].idxmax()]display(sweep.round(3))print(f"\nF1 peaks at threshold {best['threshold']} (F1 {best['f1']:.3f}), "      f"against {sweep[sweep.threshold == 0.5]['f1'].iloc[0]:.3f} at the default 0.5")

## Summary- The trivial baseline scores about 77% accuracy and an F1 of 0. That is why this  project reports F1 on the illicit class.- Both real tiers beat it comfortably on F1.- The measured difference between tier 2 and tier 3 is the answer to the research  question, and it is reported exactly as it came out.- The model's errors concentrate on addresses with very little transaction  history, which is a data limitation rather than a modelling one: with 64  transactions at most per address, some accounts simply do not give the model  enough to work with.- `src/train.py` saves the whole pipeline — scaler and classifier together — so  the application applies precisely this preprocessing and cannot drift from it.